In [ ]:
import cv2
import numpy as np
import time
import os
from picamera2 import Picamera2

# 1. Clean up any existing instances currently lingering in Jupyter's memory
for cam_var in ['cam_l', 'cam_r']:
    if cam_var in globals() and globals()[cam_var] is not None:
        try:
            globals()[cam_var].stop()
            globals()[cam_var].close()
        except Exception:
            pass
        globals()[cam_var] = None

# Create folders for the stereo calibration pairs
os.makedirs("calib_data/left", exist_ok=True)
os.makedirs("calib_data/right", exist_ok=True)

cam_l = None
cam_r = None
# 1. Define your chessboard layout based on your 25x17 cm sheet
CHECKERBOARD = (9, 6)       # 9 columns and 6 rows of inner cross-junction corners
SQUARE_SIZE_MM = 25.0       # Each square is exactly 2.5 cm (25.0 mm)

try:
    print("Acquiring IMX708 cameras...")
    res = (640, 480)
    
    # Initialize Left Camera
    cam_l = Picamera2(0)
    cfg_l = cam_l.create_preview_configuration(main={"size": res, "format": "RGB888"})
    cam_l.configure(cfg_l)
    cam_l.set_controls({"AfMode": 0, "LensPosition": 0.0}) # Manual focus infinity
    
    # Initialize Right Camera
    cam_r = Picamera2(1)
    cfg_r = cam_r.create_preview_configuration(main={"size": res, "format": "RGB888"})
    cam_r.configure(cfg_r)
    cam_r.set_controls({"AfMode": 0, "LensPosition": 0.0}) # Manual focus infinity

    # Staggered startup to avoid RP1 bus collisions
    cam_l.start()
    time.sleep(0.5)
    cam_r.start()
    time.sleep(0.5)

    print("\n=== Auto-Interval Capture Started ===")
    print("A photo will be taken every 4.0 seconds.")
    print("Hold the chessboard in front of both cameras and stay still for the snap!")
    print("To exit: focus on the preview window and press 'q'.\n")

    count = 0
    target_count = 20
    interval = 4.0
    last_capture_time = time.time()

    while count < target_count:
        img_l = cam_l.capture_array()
        img_r = cam_r.capture_array()

        # Check if left image needs rotation to match physical mounting:
        # img_l = cv2.rotate(img_l, cv2.ROTATE_180)

        current_time = time.time()
        elapsed = current_time - last_capture_time
        remaining = max(0.0, interval - elapsed)

        # Build live side-by-side preview with countdown text
        preview = np.hstack((img_l, img_r))
        cv2.putText(preview, f"Next Snap in: {remaining:.1f}s", (30, 45), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)
        cv2.putText(preview, f"Saved: {count}/{target_count}", (30, 90), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA)

        cv2.imshow("Auto Stereo Calibration Capture", preview)

        # Trigger automatic snap when timer finishes
        if elapsed >= interval:
            cv2.imwrite(f"calib_data/left/img_{count:02d}.png", cv2.cvtColor(img_l, cv2.COLOR_RGB2BGR))
            cv2.imwrite(f"calib_data/right/img_{count:02d}.png", cv2.cvtColor(img_r, cv2.COLOR_RGB2BGR))
            print(f"📸 Captured calibration pair {count:02d}/{target_count}")
            count += 1
            last_capture_time = current_time

            # Screen flash indicator
            flash = np.ones_like(preview) * 255
            cv2.imshow("Auto Stereo Calibration Capture", flash)
            cv2.waitKey(100)

        # Exit early if 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("Stopped manually by user.")
            break

    if count >= target_count:
        print("\n🎉 Completed 20/20 stereo captures!")

except Exception as e:
    print(f"\n❌ Execution Error: {e}")

finally:
    # Guaranteed cleanup of hardware pipelines
    print("Releasing camera hardware...")
    if cam_l is not None:
        try:
            cam_l.stop()
            cam_l.close()
        except Exception:
            pass
        cam_l = None
        
    if cam_r is not None:
        try:
            cam_r.stop()
            cam_r.close()
        except Exception:
            pass
        cam_r = None

    cv2.destroyAllWindows()
    print("Camera hardware safely released. Ready to run again!")


Calibration V_2

In [ ]:
import cv2
import numpy as np
import glob
import os

# 1. Define Chessboard Parameters
CHECKERBOARD = (9, 6)       
SQUARE_SIZE_MM = 25.0       

subpix_criteria = (cv2.TERM_CRITERIA_MAX_ITER + cv2.TERM_CRITERIA_EPS, 100, 1e-6)

# High-precision fisheye settings
calib_flags = (cv2.fisheye.CALIB_RECOMPUTE_EXTRINSIC + 
               cv2.fisheye.CALIB_CHECK_COND + 
               cv2.fisheye.CALIB_FIX_SKEW)

# 2. Build Object Points Matrix
objp = np.zeros((1, CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
objp[0, :, :2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2) * SQUARE_SIZE_MM

objpoints = []       
imgpoints_l = []     
imgpoints_r = []     

images_l = sorted(glob.glob("calib_data/left/*.png"))
images_r = sorted(glob.glob("calib_data/right/*.png"))

print(f"Analyzing {len(images_l)} stereo image pairs...")

valid_pairs = 0
for path_l, path_r in zip(images_l, images_r):
    img_l = cv2.imread(path_l, cv2.IMREAD_GRAYSCALE)
    img_r = cv2.imread(path_r, cv2.IMREAD_GRAYSCALE)

    flags = (cv2.CALIB_CB_ADAPTIVE_THRESH + 
             cv2.CALIB_CB_NORMALIZE_IMAGE + 
             cv2.CALIB_CB_FILTER_QUADS)

    ret_l, corners_l = cv2.findChessboardCorners(img_l, CHECKERBOARD, flags)
    ret_r, corners_r = cv2.findChessboardCorners(img_r, CHECKERBOARD, flags)

    if ret_l and ret_r:
        objpoints.append(objp)
        corners_l = cv2.cornerSubPix(img_l, corners_l, (3, 3), (-1, -1), subpix_criteria)
        corners_r = cv2.cornerSubPix(img_r, corners_r, (3, 3), (-1, -1), subpix_criteria)
        
        # Reshape the points to (1, N, 2) to prevent the "arithm_op" crash
        imgpoints_l.append(corners_l.reshape(1, -1, 2))
        imgpoints_r.append(corners_r.reshape(1, -1, 2))
        valid_pairs += 1

print(f"\nProcessed {valid_pairs}/{len(images_l)} image pairs successfully.")

if valid_pairs < 8:
    print("⚠️ Warning: Low number of calibration frames. Results may be slightly distorted.")

# Initialize the exact C++ parameters expected by the solver
K_l = np.zeros((3, 3), dtype=np.float64)
D_l = np.zeros((4, 1), dtype=np.float64)
K_r = np.zeros((3, 3), dtype=np.float64)
D_r = np.zeros((4, 1), dtype=np.float64)

# Format the points specifically as float64 list arrays for the C++ compiler
objpoints_arr = [np.array(p, dtype=np.float64) for p in objpoints]
imgpoints_l_arr = [np.array(p, dtype=np.float64) for p in imgpoints_l]
imgpoints_r_arr = [np.array(p, dtype=np.float64) for p in imgpoints_r]

R = np.zeros((3, 3), dtype=np.float64)
T = np.zeros((3, 1), dtype=np.float64)

print("Executing OpenCV wide-angle mathematical solver...")
try:
    rms, _, _, _, _, rotation, translation = cv2.fisheye.stereoCalibrate(
        objpoints_arr, imgpoints_l_arr, imgpoints_r_arr,
        K_l, D_l, K_r, D_r,
        img_l.shape[::-1], R, T,
        calib_flags, subpix_criteria
    )

    print(f"🎉 Success! RMS Reprojection Error: {rms:.4f}")
    print("Computing undistortion maps and projection matrix (Q)...")
    
    R1, R2, P1, P2, Q = cv2.fisheye.stereoRectify(
        K_l, D_l, K_r, D_r, 
        img_l.shape[::-1], 
        rotation, translation, 
        0
    )
    
    # Save matrices
    np.savez("stereo_calibration.npz", 
             K_l=K_l, D_l=D_l, 
             K_r=K_r, D_r=D_r, 
             R1=R1, R2=R2, 
             P1=P1, P2=P2, 
             Q=Q)
    print("💾 Successfully saved 'stereo_calibration.npz'!")
    
except Exception as e:
    print(f"❌ Solver failed: {e}")


run With 7 pic

In [ ]:
import cv2
import numpy as np
import time
import os
from picamera2 import Picamera2

# 1. Load your high-precision calibration matrices
if not os.path.exists("stereo_calibration.npz"):
    raise FileNotFoundError("Calibration file 'stereo_calibration.npz' not found! Run the calibration cell first.")

calib = np.load("stereo_calibration.npz")
K_l, D_l, R1, P1 = calib['K_l'], calib['D_l'], calib['R1'], calib['P1']
K_r, D_r, R2, P2 = calib['K_r'], calib['D_r'], calib['R2'], calib['P2']
Q = calib['Q']

w, h = 640, 480

# Precompute the distortion map coordinate lookup tables (makes execution fast on Pi 5)
map_l1, map_l2 = cv2.fisheye.initUndistortRectifyMap(K_l, D_l, R1, P1, (w, h), cv2.CV_16SC2)
map_r1, map_r2 = cv2.fisheye.initUndistortRectifyMap(K_r, D_r, R2, P2, (w, h), cv2.CV_16SC2)

# 2. Reset and initialize physical cameras safely
for cam_var in ['cam_l', 'cam_r']:
    if cam_var in globals() and globals()[cam_var] is not None:
        try:
            globals()[cam_var].stop()
            globals()[cam_var].close()
        except Exception:
            pass
        globals()[cam_var] = None

cam_l = None
cam_r = None

try:
    print("Acquiring stabilized camera streams...")
    cam_l = Picamera2(0)
    cfg_l = cam_l.create_preview_configuration(main={"size": (w, h), "format": "RGB888"})
    cam_l.configure(cfg_l)
    cam_l.set_controls({"AfMode": 0, "LensPosition": 0.0}) # Lock focus to infinity
    
    cam_r = Picamera2(1)
    cfg_r = cam_r.create_preview_configuration(main={"size": (w, h), "format": "RGB888"})
    cam_r.configure(cfg_r)
    cam_r.set_controls({"AfMode": 0, "LensPosition": 0.0}) # Lock focus to infinity

    cam_l.start()
    time.sleep(0.5)
    cam_r.start()
    time.sleep(0.5)

    # 3. Define the Stereo Block Matcher
    # We fine-tune this with a speckle filter and uniqueness ratio for clean, noise-free edges
    stereo = cv2.StereoSGBM_create(
        minDisparity=2,         # Minimum search range
        numDisparities=64,      # Must be divisible by 16 (controls maximum depth search depth)
        blockSize=5,            # Matcher window size (smaller = sharper details, larger = smoother)
        P1=8 * 3 * 5**2,        # Penalty parameter 1 for depth smoothing
        P2=32 * 3 * 5**2,       # Penalty parameter 2 for depth smoothing
        disp12MaxDiff=1,        # Maximum allowed difference in left-right consistency check
        uniquenessRatio=15,     # Margin in percentage by which the best cost should exceed the second best
        speckleWindowSize=100,  # Filter size to remove noise specks
        speckleRange=32         # Maximum variance allowed within a noise speck group
    )

    # PLY 3D file writer function
    def export_to_ply(filename, points_3d, colors):
        print("Processing 3D spatial points cloud...")
        # Only export points within 15 cm and 2.5 meters of the camera rig (ignores infinity background noise)
        mask = (points_3d[:, 2] > 150) & (points_3d[:, 2] < 2500) & (np.abs(points_3d[:, 0]) < 2000)
        valid_points = points_3d[mask]
        valid_colors = colors[mask]
        
        header = f"""ply
format ascii 1.0
element vertex {len(valid_points)}
property float x
property float y
property float z
property uchar red
property uchar green
property uchar blue
end_header
"""
        with open(filename, 'w') as f:
            f.write(header)
            for p, c in zip(valid_points, valid_colors):
                f.write(f"{p[0]:.2f} {p[1]:.2f} {p[2]:.2f} {int(c[0])} {int(c[1])} {int(c[2])}\n")
        print(f"🎉 3D Point Cloud successfully exported to '{filename}'!")

    print("\n=== Live 3D Depth Active ===")
    print("👉 Keyboard controls (focus on either image window first):")
    print("   • Press 'p' to export a 3D Point Cloud (.PLY)")
    print("   • Press 'q' to safely quit and release hardware.")

    while True:
        frame_l = cam_l.capture_array()
        frame_r = cam_r.capture_array()

        # Apply Real-Time Rectification (makes curved wide-angle images straight and aligned)
        rect_l = cv2.remap(frame_l, map_l1, map_l2, cv2.INTER_LINEAR)
        rect_r = cv2.remap(frame_r, map_r1, map_r2, cv2.INTER_LINEAR)

        # Compute depth map
        gray_l = cv2.cvtColor(rect_l, cv2.COLOR_RGB2GRAY)
        gray_r = cv2.cvtColor(rect_r, cv2.COLOR_RGB2GRAY)
        disparity = stereo.compute(gray_l, gray_r).astype(np.float32) / 16.0

        # Create high-contrast colormap for depth visualization
        disp_vis = cv2.normalize(disparity, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
        disp_color = cv2.applyColorMap(disp_vis, cv2.COLORMAP_JET)

        # Display output windows
        rect_side_by_side = np.hstack((rect_l, rect_r))
        cv2.imshow("Rectified Streams (Left | Right)", rect_side_by_side)
        cv2.imshow("Live 3D Depth Map", disp_color)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('p'):
            # Translate the 2D pixel displacement into physical (X, Y, Z) coordinates using Q matrix
            points_3d = cv2.reprojectImageTo3D(disparity, Q)
            # Reorganize 2D matrix structure to simple 3D coordinate lists
            points_flat = points_3d.reshape(-1, 3)
            # Map colors from the rectified left frame to the points
            colors_flat = cv2.cvtColor(rect_l, cv2.COLOR_RGB2BGR).reshape(-1, 3)
            
            export_to_ply("reconstruction_3d.ply", points_flat, colors_flat)
            
        elif key == ord('q'):
            print("Exit command received. Shutting down...")
            break

except Exception as e:
    print(f"\n❌ Execution Error: {e}")

finally:
    # Guaranteed cleanup of hardware
    print("Releasing camera hardware...")
    if cam_l is not None:
        try:
            cam_l.stop()
            cam_l.close()
        except Exception:
            pass
        cam_l = None
        
    if cam_r is not None:
        try:
            cam_r.stop()
            cam_r.close()
        except Exception:
            pass
        cam_r = None

    cv2.destroyAllWindows()
    print("Camera hardware safely released. Finished!")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def load_ply(filename):
    """Loads points and colors from a standard ASCII PLY file."""
    points = []
    colors = []
    header_ended = False
    
    with open(filename, 'r') as f:
        for line in f:
            if "end_header" in line:
                header_ended = True
                continue
            if header_ended:
                parts = line.strip().split()
                if len(parts) >= 6:
                    # x, y, z, r, g, b
                    points.append([float(parts[0]), float(parts[1]), float(parts[2])])
                    colors.append([int(parts[3])/255.0, int(parts[4])/255.0, int(parts[5])/255.0])
                    
    return np.array(points), np.array(colors)

print("Loading 3D point cloud...")
points, colors = load_ply("reconstruction_3d.ply")

# Downsample points so the plot remains fast and responsive
downsample_rate = 20  # Display 1 out of every 20 points
ds_points = points[::downsample_rate]
ds_colors = colors[::downsample_rate]

print(f"Rendering {len(ds_points)} points in 3D...")

# Enable interactive rotation in Jupyter Notebook
%matplotlib notebook  

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Scatter plot using real colors captured by your IMX708 cameras
ax.scatter(ds_points[:, 0], ds_points[:, 2], -ds_points[:, 1], 
           c=ds_colors, s=1, marker='.')

ax.set_xlabel('X (Width)')
ax.set_ylabel('Z (Depth/Distance)')
ax.set_zlabel('Y (Height)')
ax.set_title("3D Stereo Point Cloud (Drag with your mouse to rotate!)")

# Limit view to clean up infinite background noise
ax.set_xlim(-1000, 1000)
ax.set_ylim(200, 2500)  # Shows objects between 20cm and 2.5 meters
ax.set_zlim(-1000, 1000)

plt.show()


In [ ]:
!pip install ultralytics --break-system-packages


In [6]:
import cv2
import numpy as np
import time
import os
from picamera2 import Picamera2
from ultralytics import YOLO

# 1. Load Calibration Matrices
if not os.path.exists("stereo_calibration.npz"):
    raise FileNotFoundError("Missing 'stereo_calibration.npz'. Run calibration first!")

calib = np.load("stereo_calibration.npz")
K_l, D_l, R1, P1 = calib['K_l'], calib['D_l'], calib['R1'], calib['P1']
K_r, D_r, R2, P2 = calib['K_r'], calib['D_r'], calib['R2'], calib['P2']
Q = calib['Q']

w, h = 640, 480
map_l1, map_l2 = cv2.fisheye.initUndistortRectifyMap(K_l, D_l, R1, P1, (w, h), cv2.CV_16SC2)
map_r1, map_r2 = cv2.fisheye.initUndistortRectifyMap(K_r, D_r, R2, P2, (w, h), cv2.CV_16SC2)

# 2. Load Local YOLOv8 Pose Model
print("Loading pose model...")
model = YOLO("yolov8n-pose.pt")

# 3. Clean up and start cameras
for cam_var in ['cam_l', 'cam_r']:
    if cam_var in globals() and globals()[cam_var] is not None:
        try:
            globals()[cam_var].stop(); globals()[cam_var].close()
        except Exception: pass
        globals()[cam_var] = None

cam_l = Picamera2(0)
cam_r = Picamera2(1)
cam_l.configure(cam_l.create_preview_configuration(main={"size": (w, h), "format": "RGB888"}))
cam_r.configure(cam_r.create_preview_configuration(main={"size": (w, h), "format": "RGB888"}))
cam_l.set_controls({"AfMode": 0, "LensPosition": 0.0})
cam_r.set_controls({"AfMode": 0, "LensPosition": 0.0})
cam_l.start(); time.sleep(0.5); cam_r.start(); time.sleep(0.5)

# High-Precision Stereo Matcher
stereo = cv2.StereoSGBM_create(
    minDisparity=2, numDisparities=64, blockSize=5,
    P1=8 * 3 * 25, P2=32 * 3 * 25, disp12MaxDiff=1,
    uniquenessRatio=15, speckleWindowSize=100, speckleRange=32
)

def get_clean_3d_point(points_3d, px, py, window_size=5):
    """
    Extracts a highly stable 3D coordinate from a neighborhood patch
    around a joint pixel to filter out depth sensor noise.
    """
    half_w = window_size // 2
    # Define patch boundaries safely within frame limits
    y_min, y_max = max(0, py - half_w), min(h - 1, py + half_w + 1)
    x_min, x_max = max(0, px - half_w), min(w - 1, px + half_w + 1)
    
    # Extract the neighborhood 3D points
    patch = points_3d[y_min:y_max, x_min:x_max]
    
    # Reshape patch to list of (X, Y, Z) points
    pts = patch.reshape(-1, 3)
    
    # Filter out points with invalid, nan, or infinite depths
    valid_mask = (pts[:, 2] > 100) & (pts[:, 2] < 2500) & (~np.isnan(pts[:, 2])) & (~np.isinf(pts[:, 2]))
    valid_pts = pts[valid_mask]
    
    if len(valid_pts) > 0:
        # Use median coordinates of the patch to eliminate outliers
        return np.median(valid_pts, axis=0)
    else:
        return None

def compute_3d_angle(pt_hip, pt_knee, pt_ankle):
    """Calculates true spatial angle using 3D Euclidean vectors."""
    v1 = pt_hip - pt_knee     # Thigh vector (pointing from knee to hip)
    v2 = pt_ankle - pt_knee   # Shin vector (pointing from knee to ankle)
    
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    if norm1 == 0 or norm2 == 0:
        return None
        
    cosine = np.dot(v1, v2) / (norm1 * norm2)
    angle = np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))
    return angle

print("\n=== High-Precision 3D Knee Angle Tracker Active ===")
print("Tracking spatial coordinates directly from the stereo model.")
print("Press 'q' on the video window to quit.")

try:
    while True:
        frame_l = cam_l.capture_array()
        frame_r = cam_r.capture_array()

        # Rectify feeds (straighten wide-angle barrel distortion)
        rect_l = cv2.remap(frame_l, map_l1, map_l2, cv2.INTER_LINEAR)
        rect_r = cv2.remap(frame_r, map_r1, map_r2, cv2.INTER_LINEAR)

        # Compute 3D spatial point matrix (X, Y, Z coordinates in millimeters)
        gray_l = cv2.cvtColor(rect_l, cv2.COLOR_RGB2GRAY)
        gray_r = cv2.cvtColor(rect_r, cv2.COLOR_RGB2GRAY)
        disparity = stereo.compute(gray_l, gray_r).astype(np.float32) / 16.0
        points_3d = cv2.reprojectImageTo3D(disparity, Q)

        # Run Pose Estimation on the undistorted left camera feed
        results = model(rect_l, verbose=False, conf=0.5)
        annotated_frame = rect_l.copy()

        for result in results:
            if result.keypoints is not None and len(result.keypoints.xy) > 0:
                kpts = result.keypoints.xy[0].cpu().numpy()
                
                if len(kpts) >= 17:
                    # DIRECT INDEX SPLICING FIX: Extract the actual coordinate arrays
                    # COCO Indices: 12=Right Hip, 14=Right Knee, 16=Right Ankle
                    r_hip = kpts[12]
                    r_knee = kpts[14]
                    r_ankle = kpts[16]
                    
                    # Verify these specific joints are detected (non-zero values)
                    if r_hip[0] > 0 and r_knee[0] > 0 and r_ankle[0] > 0:
                        kx, ky = int(r_knee[0]), int(r_knee[1])
                        hx, hy = int(r_hip[0]), int(r_hip[1])
                        ax, ay = int(r_ankle[0]), int(r_ankle[1])

                        # Draw skeletal connections in 2D space
                        cv2.line(annotated_frame, (hx, hy), (kx, ky), (0, 255, 255), 3)
                        cv2.line(annotated_frame, (kx, ky), (ax, ay), (0, 255, 255), 3)
                        cv2.circle(annotated_frame, (kx, ky), 6, (0, 0, 255), -1)

                        # Extract stable, noise-filtered 3D positions in millimeters
                        p3_hip = get_clean_3d_point(points_3d, hx, hy)
                        p3_knee = get_clean_3d_point(points_3d, kx, ky)
                        p3_ankle = get_clean_3d_point(points_3d, ax, ay)

                        # Ensure all three joint positions have a valid lock from the 3D model
                        if p3_hip is not None and p3_knee is not None and p3_ankle is not None:
                            # Calculate the true spatial 3D angle
                            angle_3d = compute_3d_angle(p3_hip, p3_knee, p3_ankle)
                            
                            if angle_3d is not None:
                                # Print precise coordinates to the terminal for debugging (Z divided by 10 for cm)
                                print(f"Hip Z: {p3_hip[2]/10.0:.1f}cm | Knee Z: {p3_knee[2]/10.0:.1f}cm | Ankle Z: {p3_ankle[2]/10.0:.1f}cm | Angle: {angle_3d:.2f}°")
                                
                                # Overlay the precise 3D angle on the frame
                                cv2.putText(annotated_frame, f"Knee 3D: {angle_3d:.1f} deg", (kx - 65, ky - 25),
                                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                        else:
                            # Warn user if the 3D spatial model loses sight of the depth details
                            cv2.putText(annotated_frame, "3D Scan: Locking Depth...", (kx - 80, ky - 25),
                                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

        cv2.imshow("Knee Angle 3D Gait Tracker", annotated_frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

except Exception as e:
    print(f"\n❌ Loop interrupted: {e}")

finally:
    cam_l.stop(); cam_l.close()
    cam_r.stop(); cam_r.close()
    cv2.destroyAllWindows()
    print("Safely released camera resources.")


Loading pose model...


[3:46:20.951424014] [6669]  INFO Camera camera_manager.cpp:340 libcamera v0.7.1+rpt20260429
[3:46:20.965403487] [7007]  INFO RPI pisp.cpp:720 libpisp version v1.5.0 01-05-2026 (05:33:53)
[3:46:20.966011592] [7007]  WARN CameraSensorProperties camera_sensor_properties.cpp:548 No static properties available for 'imx708_wide'
[3:46:20.966019833] [7007]  WARN CameraSensorProperties camera_sensor_properties.cpp:550 Please consider updating the camera sensor properties database
[3:46:20.966904935] [7007]  INFO IPAProxy ipa_proxy.cpp:184 Using tuning file /usr/share/libcamera/ipa/rpi/pisp/imx708_wide.json
[3:46:20.974180215] [7007]  WARN CameraSensor camera_sensor_legacy.cpp:502 'imx708_wide': No sensor delays found in static properties. Assuming unverified defaults.
[3:46:20.974561952] [7007]  INFO Camera camera_manager.cpp:223 Adding camera '/base/axi/pcie@1000120000/rp1/i2c@88000/imx708@1a' for pipeline handler rpi/pisp
[3:46:20.974568545] [7007]  INFO RPI pisp.cpp:1181 Registered camera /


=== High-Precision 3D Knee Angle Tracker Active ===
Tracking spatial coordinates directly from the stereo model.
Press 'q' on the video window to quit.
Hip Z: 24.0cm | Knee Z: 24.0cm | Ankle Z: 24.0cm | Angle: 168.88°
Hip Z: 24.0cm | Knee Z: 24.0cm | Ankle Z: 24.0cm | Angle: 170.65°
Hip Z: 25.9cm | Knee Z: 24.0cm | Ankle Z: 24.0cm | Angle: 171.47°
Hip Z: 25.9cm | Knee Z: 24.0cm | Ankle Z: 24.0cm | Angle: 173.18°
Hip Z: 24.0cm | Knee Z: 24.0cm | Ankle Z: 24.0cm | Angle: 167.72°
Hip Z: 24.0cm | Knee Z: 24.0cm | Ankle Z: 24.1cm | Angle: 169.21°
Hip Z: 25.9cm | Knee Z: 24.0cm | Ankle Z: 24.0cm | Angle: 165.48°
Hip Z: 24.0cm | Knee Z: 25.9cm | Ankle Z: 24.0cm | Angle: 109.67°
Hip Z: 24.0cm | Knee Z: 24.0cm | Ankle Z: 24.0cm | Angle: 155.09°
Hip Z: 24.0cm | Knee Z: 24.0cm | Ankle Z: 24.0cm | Angle: 151.48°
Hip Z: 24.0cm | Knee Z: 24.0cm | Ankle Z: 24.0cm | Angle: 158.98°
Hip Z: 24.0cm | Knee Z: 24.0cm | Ankle Z: 24.0cm | Angle: 164.59°
Hip Z: 24.0cm | Knee Z: 24.0cm | Ankle Z: 24.0cm | Angl

Hip Z: 24.0cm | Knee Z: 24.0cm | Ankle Z: 24.0cm | Angle: 147.09°
Hip Z: 24.0cm | Knee Z: 24.0cm | Ankle Z: 24.0cm | Angle: 149.76°
Hip Z: 24.0cm | Knee Z: 24.0cm | Ankle Z: 24.0cm | Angle: 147.68°
Hip Z: 24.0cm | Knee Z: 24.0cm | Ankle Z: 24.0cm | Angle: 141.45°
Hip Z: 24.0cm | Knee Z: 24.0cm | Ankle Z: 24.0cm | Angle: 141.24°
Hip Z: 24.0cm | Knee Z: 24.0cm | Ankle Z: 24.0cm | Angle: 134.18°
Hip Z: 25.9cm | Knee Z: 24.0cm | Ankle Z: 24.0cm | Angle: 139.52°
Hip Z: 24.0cm | Knee Z: 24.0cm | Ankle Z: 25.9cm | Angle: 129.45°
Hip Z: 24.0cm | Knee Z: 24.0cm | Ankle Z: 25.8cm | Angle: 126.35°
Hip Z: 24.0cm | Knee Z: 24.0cm | Ankle Z: 25.9cm | Angle: 143.97°
Hip Z: 24.0cm | Knee Z: 24.0cm | Ankle Z: 24.0cm | Angle: 139.07°
Hip Z: 24.0cm | Knee Z: 24.0cm | Ankle Z: 24.0cm | Angle: 131.68°
Hip Z: 24.0cm | Knee Z: 24.0cm | Ankle Z: 24.0cm | Angle: 136.68°
Hip Z: 24.0cm | Knee Z: 24.0cm | Ankle Z: 25.9cm | Angle: 136.73°
Hip Z: 24.0cm | Knee Z: 24.0cm | Ankle Z: 24.0cm | Angle: 158.05°
Hip Z: 24.

In [7]:
import cv2
import numpy as np
import time
import os
from picamera2 import Picamera2
from ultralytics import YOLO

# 1. Load Calibration Matrices
if not os.path.exists("stereo_calibration.npz"):
    raise FileNotFoundError("Missing 'stereo_calibration.npz'. Run calibration first!")

calib = np.load("stereo_calibration.npz")
K_l, D_l, R1, P1 = calib['K_l'], calib['D_l'], calib['R1'], calib['P1']
K_r, D_r, R2, P2 = calib['K_r'], calib['D_r'], calib['R2'], calib['P2']
Q = calib['Q']

w, h = 640, 480
map_l1, map_l2 = cv2.fisheye.initUndistortRectifyMap(K_l, D_l, R1, P1, (w, h), cv2.CV_16SC2)
map_r1, map_r2 = cv2.fisheye.initUndistortRectifyMap(K_r, D_r, R2, P2, (w, h), cv2.CV_16SC2)

# 2. Load Local YOLOv8 Pose Model
print("Loading pose model...")
model = YOLO("yolov8n-pose.pt")

# 3. Clean up and start cameras safely
for cam_var in ['cam_l', 'cam_r']:
    if cam_var in globals() and globals()[cam_var] is not None:
        try:
            globals()[cam_var].stop(); globals()[cam_var].close()
        except Exception: pass
        globals()[cam_var] = None

cam_l = Picamera2(0)
cam_r = Picamera2(1)
cam_l.configure(cam_l.create_preview_configuration(main={"size": (w, h), "format": "RGB888"}))
cam_r.configure(cam_r.create_preview_configuration(main={"size": (w, h), "format": "RGB888"}))
cam_l.set_controls({"AfMode": 0, "LensPosition": 0.0})
cam_r.set_controls({"AfMode": 0, "LensPosition": 0.0})
cam_l.start(); time.sleep(0.5); cam_r.start(); time.sleep(0.5)

# High-Precision Stereo Matcher with wider disparity search
stereo = cv2.StereoSGBM_create(
    minDisparity=0,
    numDisparities=80,          # Divisible by 16; broader range to catch closer legs
    blockSize=7,                # Slightly larger window captures limb texture better
    P1=8 * 3 * 7**2,
    P2=32 * 3 * 7**2,
    disp12MaxDiff=2,
    uniquenessRatio=10,         # Slightly relaxed to prevent depth holes on flat cloth
    speckleWindowSize=50,
    speckleRange=16
)

def get_robust_3d_point(points_3d, px, py):
    """
    Adaptive hierarchical sampling: starts at 5x5 window and expands to 19x19
    to eliminate 'Locking Depth' dropouts caused by textureless clothing.
    """
    for window_size in [5, 11, 19]:
        half_w = window_size // 2
        y_min, y_max = max(0, py - half_w), min(h - 1, py + half_w + 1)
        x_min, x_max = max(0, px - half_w), min(w - 1, px + half_w + 1)
        
        patch = points_3d[y_min:y_max, x_min:x_max].reshape(-1, 3)
        # Accept valid depths between 20 cm and 3.5 meters
        valid = (patch[:, 2] > 200) & (patch[:, 2] < 3500) & (~np.isnan(patch[:, 2])) & (~np.isinf(patch[:, 2]))
        valid_pts = patch[valid]
        
        if len(valid_pts) >= 3:
            return np.median(valid_pts, axis=0)
    return None

def compute_3d_angle(pt_hip, pt_knee, pt_ankle):
    """Calculates true spatial angle in 3D Euclidean space."""
    v1 = pt_hip - pt_knee
    v2 = pt_ankle - pt_knee
    norm1, norm2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if norm1 == 0 or norm2 == 0:
        return None
    cosine = np.dot(v1, v2) / (norm1 * norm2)
    return float(np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0))))

print("\n=== Dual-Leg 3D Gait Tracker Active ===")
print("Tracking Left and Right Knee Angles simultaneously.")
print("Press 'q' on the video window to quit.")

try:
    while True:
        frame_l = cam_l.capture_array()
        frame_r = cam_r.capture_array()

        rect_l = cv2.remap(frame_l, map_l1, map_l2, cv2.INTER_LINEAR)
        rect_r = cv2.remap(frame_r, map_r1, map_r2, cv2.INTER_LINEAR)

        gray_l = cv2.cvtColor(rect_l, cv2.COLOR_RGB2GRAY)
        gray_r = cv2.cvtColor(rect_r, cv2.COLOR_RGB2GRAY)
        disparity = stereo.compute(gray_l, gray_r).astype(np.float32) / 16.0
        points_3d = cv2.reprojectImageTo3D(disparity, Q)

        results = model(rect_l, verbose=False, conf=0.4)
        annotated_frame = rect_l.copy()

        # HUD background banner
        cv2.rectangle(annotated_frame, (10, 10), (330, 90), (20, 20, 20), -1)
        cv2.putText(annotated_frame, "GAIT REHAB TRACKER (3D)", (20, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)

        l_angle_str = "Left:  N/A"
        r_angle_str = "Right: N/A"

        for result in results:
            if result.keypoints is not None and len(result.keypoints.xy) > 0:
                kpts = result.keypoints.xy[0].cpu().numpy()
                
                if len(kpts) >= 17:
                    # Leg configurations: (Hip_idx, Knee_idx, Ankle_idx, LineColor, Label)
                    legs = [
                        (11, 13, 15, (255, 200, 0), "L"),   # Left Leg (Cyan/Yellow)
                        (12, 14, 16, (0, 220, 255), "R")    # Right Leg (Yellow/Orange)
                    ]

                    for hip_idx, knee_idx, ankle_idx, color, side in legs:
                        hip, knee, ankle = kpts[hip_idx], kpts[knee_idx], kpts[ankle_idx]

                        if hip[0] > 0 and knee[0] > 0 and ankle[0] > 0:
                            hx, hy = int(hip[0]), int(hip[1])
                            kx, ky = int(knee[0]), int(knee[1])
                            ax, ay = int(ankle[0]), int(ankle[1])

                            # Draw skeleton segments
                            cv2.line(annotated_frame, (hx, hy), (kx, ky), color, 3)
                            cv2.line(annotated_frame, (kx, ky), (ax, ay), color, 3)
                            cv2.circle(annotated_frame, (kx, ky), 5, (0, 0, 255), -1)
                            cv2.circle(annotated_frame, (hx, hy), 4, (0, 255, 0), -1)
                            cv2.circle(annotated_frame, (ax, ay), 4, (0, 255, 0), -1)

                            # Extract robust 3D points
                            p3_hip = get_robust_3d_point(points_3d, hx, hy)
                            p3_knee = get_robust_3d_point(points_3d, kx, ky)
                            p3_ankle = get_robust_3d_point(points_3d, ax, ay)

                            if p3_hip is not None and p3_knee is not None and p3_ankle is not None:
                                angle_3d = compute_3d_angle(p3_hip, p3_knee, p3_ankle)
                                if angle_3d is not None:
                                    dist_cm = p3_knee[2] / 10.0
                                    text_disp = f"{side}: {angle_3d:.1f} deg"
                                    
                                    # Update HUD text
                                    if side == "L":
                                        l_angle_str = f"Left:  {angle_3d:.1f}° ({dist_cm:.0f}cm)"
                                    else:
                                        r_angle_str = f"Right: {angle_3d:.1f}° ({dist_cm:.0f}cm)"

                                    # Overlay on limb
                                    cv2.putText(annotated_frame, f"{angle_3d:.0f} deg", (kx - 25, ky - 10),
                                                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2)
                            else:
                                # Fallback indicator when depth is occluded
                                cv2.putText(annotated_frame, f"{side}: Occluded", (kx - 35, ky - 10),
                                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 255), 1)

        # Render HUD values
        cv2.putText(annotated_frame, l_angle_str, (20, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 200, 0), 1)
        cv2.putText(annotated_frame, r_angle_str, (20, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 220, 255), 1)

        cv2.imshow("Dual-Leg 3D Gait Tracker", annotated_frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

except Exception as e:
    print(f"\n❌ Error: {e}")

finally:
    cam_l.stop(); cam_l.close()
    cam_r.stop(); cam_r.close()
    cv2.destroyAllWindows()
    print("Safely released camera resources.")


Loading pose model...


[3:51:42.627413240] [7015]  INFO Camera camera_manager.cpp:340 libcamera v0.7.1+rpt20260429
[3:51:42.641816225] [32558]  INFO RPI pisp.cpp:720 libpisp version v1.5.0 01-05-2026 (05:33:53)
[3:51:42.642477070] [32558]  WARN CameraSensorProperties camera_sensor_properties.cpp:548 No static properties available for 'imx708_wide'
[3:51:42.642488996] [32558]  WARN CameraSensorProperties camera_sensor_properties.cpp:550 Please consider updating the camera sensor properties database
[3:51:42.643578985] [32558]  INFO IPAProxy ipa_proxy.cpp:184 Using tuning file /usr/share/libcamera/ipa/rpi/pisp/imx708_wide.json
[3:51:42.651174001] [32558]  WARN CameraSensor camera_sensor_legacy.cpp:502 'imx708_wide': No sensor delays found in static properties. Assuming unverified defaults.
[3:51:42.651568516] [32558]  INFO Camera camera_manager.cpp:223 Adding camera '/base/axi/pcie@1000120000/rp1/i2c@88000/imx708@1a' for pipeline handler rpi/pisp
[3:51:42.651578793] [32558]  INFO RPI pisp.cpp:1181 Registered c


=== Dual-Leg 3D Gait Tracker Active ===
Tracking Left and Right Knee Angles simultaneously.
Press 'q' on the video window to quit.
Safely released camera resources.


In [9]:
import cv2
import numpy as np
import time
import os
from picamera2 import Picamera2
from ultralytics import YOLO

# 1. Load Calibration Matrices
if not os.path.exists("stereo_calibration.npz"):
    raise FileNotFoundError("Missing 'stereo_calibration.npz'. Run calibration first!")

calib = np.load("stereo_calibration.npz")
K_l, D_l, R1, P1 = calib['K_l'], calib['D_l'], calib['R1'], calib['P1']
K_r, D_r, R2, P2 = calib['K_r'], calib['D_r'], calib['R2'], calib['P2']
Q = calib['Q']

w, h = 640, 480
map_l1, map_l2 = cv2.fisheye.initUndistortRectifyMap(K_l, D_l, R1, P1, (w, h), cv2.CV_16SC2)
map_r1, map_r2 = cv2.fisheye.initUndistortRectifyMap(K_r, D_r, R2, P2, (w, h), cv2.CV_16SC2)

# 2. Load Local YOLOv8 Pose Model
print("Loading pose model...")
model = YOLO("yolov8n-pose.pt")

# 3. Clean up and start cameras safely
for cam_var in ['cam_l', 'cam_r']:
    if cam_var in globals() and globals()[cam_var] is not None:
        try:
            globals()[cam_var].stop(); globals()[cam_var].close()
        except Exception: pass
        globals()[cam_var] = None

cam_l = Picamera2(0)
cam_r = Picamera2(1)
cam_l.configure(cam_l.create_preview_configuration(main={"size": (w, h), "format": "RGB888"}))
cam_r.configure(cam_r.create_preview_configuration(main={"size": (w, h), "format": "RGB888"}))
cam_l.set_controls({"AfMode": 0, "LensPosition": 0.0})
cam_r.set_controls({"AfMode": 0, "LensPosition": 0.0})
cam_l.start(); time.sleep(0.5); cam_r.start(); time.sleep(0.5)

# High-Precision Stereo Matcher
stereo = cv2.StereoSGBM_create(
    minDisparity=0,
    numDisparities=80,
    blockSize=7,
    P1=8 * 3 * 49,
    P2=32 * 3 * 49,
    disp12MaxDiff=2,
    uniquenessRatio=10,
    speckleWindowSize=50,
    speckleRange=16
)

# Smoothness buffer
angle_history = {"L": None, "R": None}
SMOOTH_FACTOR = 0.35

def get_foreground_3d_joint(points_3d, px, py, ref_depth=None, window=11):
    """Extracts the median foreground depth coordinate around a pixel joint."""
    half = window // 2
    y0, y1 = max(0, py - half), min(h - 1, py + half + 1)
    x0, x1 = max(0, px - half), min(w - 1, px + half + 1)

    pts = points_3d[y0:y1, x0:x1].reshape(-1, 3)
    mask = (pts[:, 2] > 150) & (pts[:, 2] < 3500) & (~np.isnan(pts[:, 2])) & (~np.isinf(pts[:, 2]))
    pts = pts[mask]

    if len(pts) == 0:
        return None

    if ref_depth is not None:
        pts = pts[np.abs(pts[:, 2] - ref_depth) < 350] # Expanded limit to 35cm
        if len(pts) == 0:
            return None

    z_sorted = np.sort(pts[:, 2])
    foreground_cutoff = z_sorted[max(0, int(len(z_sorted) * 0.3))]
    foreground_pts = pts[pts[:, 2] <= foreground_cutoff + 50]

    return np.median(foreground_pts, axis=0) if len(foreground_pts) > 0 else np.median(pts, axis=0)

def compute_spatial_angle(hip_3d, knee_3d, ankle_3d):
    """Calculates true spatial angle in 3D Euclidean space with loose bone length validation."""
    v_thigh = hip_3d - knee_3d
    v_shin = ankle_3d - knee_3d

    len_thigh = np.linalg.norm(v_thigh)
    len_shin = np.linalg.norm(v_shin)

    # Relaxed validation for close up testing (15 cm to 90 cm bone lengths)
    if not (150 < len_thigh < 900) or not (150 < len_shin < 900):
        return None, len_thigh, len_shin

    cosine = np.dot(v_thigh, v_shin) / (len_thigh * len_shin)
    raw_angle = float(np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0))))
    return raw_angle, len_thigh, len_shin

print("\n=== Robust 3D & 2D Gait Tracker Active ===")
print("Lowered person-detection limits. Real-time fallbacks enabled.")
print("Press 'q' on the video window to safely quit.")

try:
    while True:
        frame_l = cam_l.capture_array()
        frame_r = cam_r.capture_array()

        rect_l = cv2.remap(frame_l, map_l1, map_l2, cv2.INTER_LINEAR)
        rect_r = cv2.remap(frame_r, map_r1, map_r2, cv2.INTER_LINEAR)

        gray_l = cv2.cvtColor(rect_l, cv2.COLOR_RGB2GRAY)
        gray_r = cv2.cvtColor(rect_r, cv2.COLOR_RGB2GRAY)
        disparity = stereo.compute(gray_l, gray_r).astype(np.float32) / 16.0
        points_3d = cv2.reprojectImageTo3D(disparity, Q)

        # Lowered confidence threshold (conf=0.25) to detect partial legs/close ups
        results = model(rect_l, verbose=False, conf=0.25)
        annotated_frame = rect_l.copy()

        # HUD background banner
        cv2.rectangle(annotated_frame, (10, 10), (320, 85), (25, 25, 25), -1)
        cv2.putText(annotated_frame, "3D ROTATION-INVARIANT TRACKER", (20, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)

        l_hud, r_hud = "Left:  --", "Right: --"

        for result in results:
            if result.keypoints is not None and len(result.keypoints.xy) > 0:
                kpts = result.keypoints.xy[0].cpu().numpy()

                if len(kpts) >= 17:
                    legs = [
                        (11, 13, 15, (255, 210, 0), "L"), # Cyan Left leg
                        (12, 14, 16, (0, 215, 255), "R")  # Yellow Right leg
                    ]

                    for h_idx, k_idx, a_idx, col, side in legs:
                        h_pt, k_pt, a_pt = kpts[h_idx], kpts[k_idx], kpts[a_idx]

                        # Joint visibility checks
                        if h_pt[0] > 0 and k_pt[0] > 0 and a_pt[0] > 0:
                            hx, hy = int(h_pt[0]), int(h_pt[1])
                            kx, ky = int(k_pt[0]), int(k_pt[1])
                            ax, ay = int(a_pt[0]), int(a_pt[1])

                            # 1. 2D Fallback Calculation
                            v1_2d = h_pt - k_pt
                            v2_2d = a_pt - k_pt
                            cos_2d = np.dot(v1_2d, v2_2d) / (np.linalg.norm(v1_2d) * np.linalg.norm(v2_2d) + 1e-6)
                            angle_2d = np.degrees(np.arccos(np.clip(cos_2d, -1.0, 1.0)))

                            # 2. Extract High-Precision 3D Coordinates
                            p3_knee = get_foreground_3d_joint(points_3d, kx, ky, ref_depth=None)
                            p3_hip = None
                            p3_ankle = None
                            angle_3d = None
                            status_msg = "2D Active (Depth Locking...)"
                            text_color = (0, 255, 255) # Yellow for 2D fallback

                            if p3_knee is not None:
                                knee_z = p3_knee[2]
                                p3_hip = get_foreground_3d_joint(points_3d, hx, hy, ref_depth=knee_z)
                                p3_ankle = get_foreground_3d_joint(points_3d, ax, ay, ref_depth=knee_z)

                                if p3_hip is not None and p3_ankle is not None:
                                    angle_3d, lt, ls = compute_spatial_angle(p3_hip, p3_knee, p3_ankle)
                                    
                                    if angle_3d is not None:
                                        status_msg = "3D Stereo Locked"
                                        text_color = (0, 255, 0) # Green for high-precision 3D
                                        
                                        # Smooth out the 3D angle over time
                                        if angle_history[side] is None:
                                            angle_history[side] = angle_3d
                                        else:
                                            angle_history[side] = (SMOOTH_FACTOR * angle_3d + (1 - SMOOTH_FACTOR) * angle_history[side])
                                        angle_display = angle_history[side]
                                    else:
                                        # Print loose bone measurement rejects for debugging
                                        print(f"[{side}] Bone Length check rejected: Thigh={lt/10.0:.1f}cm, Shin={ls/10.0:.1f}cm")

                            # Choose what angle to display (smoothed 3D, raw 3D, or 2D fallback)
                            active_angle = angle_history[side] if (angle_history[side] is not None and angle_3d is not None) else angle_2d
                            angle_source_label = "3D" if angle_3d is not None else "2D"
                            dist_cm = (p3_knee[2] / 10.0) if p3_knee is not None else 0.0

                            # Update HUD
                            hud_txt = f"{side} ({angle_source_label}): {active_angle:.1f}° ({dist_cm:.0f}cm)"
                            if side == "L": l_hud = hud_txt
                            else: r_hud = hud_txt

                            # Draw skeletal connections on the screen
                            cv2.line(annotated_frame, (hx, hy), (kx, ky), col, 3)
                            cv2.line(annotated_frame, (kx, ky), (ax, ay), col, 3)
                            cv2.circle(annotated_frame, (kx, ky), 6, (0, 0, 255), -1)
                            cv2.circle(annotated_frame, (hx, hy), 4, (0, 255, 0), -1)
                            cv2.circle(annotated_frame, (ax, ay), 4, (0, 255, 0), -1)
                            
                            # Overlay joint angle
                            cv2.putText(annotated_frame, f"{active_angle:.0f}° ({angle_source_label})", (kx - 30, ky - 12),
                                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, text_color, 2)

        # Render HUD Banner text
        cv2.putText(annotated_frame, l_hud, (20, 52), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 210, 0), 1)
        cv2.putText(annotated_frame, r_hud, (20, 72), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 215, 255), 1)

        cv2.imshow("Rotation-Invariant 3D Gait Tracker", annotated_frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

except Exception as e:
    print(f"\n❌ Error during execution: {e}")

finally:
    print("Safely releasing camera hardware...")
    cam_l.stop(); cam_l.close()
    cam_r.stop(); cam_r.close()
    cv2.destroyAllWindows()
    print("Finished cleanup!")


Loading pose model...


[4:13:03.169043919] [55723]  INFO Camera camera_manager.cpp:340 libcamera v0.7.1+rpt20260429
[4:13:03.182541817] [109755]  INFO RPI pisp.cpp:720 libpisp version v1.5.0 01-05-2026 (05:33:53)
[4:13:03.183205478] [109755]  WARN CameraSensorProperties camera_sensor_properties.cpp:548 No static properties available for 'imx708_wide'
[4:13:03.183215737] [109755]  WARN CameraSensorProperties camera_sensor_properties.cpp:550 Please consider updating the camera sensor properties database
[4:13:03.184171451] [109755]  INFO IPAProxy ipa_proxy.cpp:184 Using tuning file /usr/share/libcamera/ipa/rpi/pisp/imx708_wide.json
[4:13:03.191753828] [109755]  WARN CameraSensor camera_sensor_legacy.cpp:502 'imx708_wide': No sensor delays found in static properties. Assuming unverified defaults.
[4:13:03.192173157] [109755]  INFO Camera camera_manager.cpp:223 Adding camera '/base/axi/pcie@1000120000/rp1/i2c@88000/imx708@1a' for pipeline handler rpi/pisp
[4:13:03.192184491] [109755]  INFO RPI pisp.cpp:1181 Regi


=== Robust 3D & 2D Gait Tracker Active ===
Lowered person-detection limits. Real-time fallbacks enabled.
Press 'q' on the video window to safely quit.
[L] Bone Length check rejected: Thigh=4.2cm, Shin=2.6cm
[R] Bone Length check rejected: Thigh=2.2cm, Shin=11.5cm
[L] Bone Length check rejected: Thigh=7.3cm, Shin=3.8cm
[R] Bone Length check rejected: Thigh=6.5cm, Shin=10.2cm
[L] Bone Length check rejected: Thigh=6.6cm, Shin=2.2cm
[R] Bone Length check rejected: Thigh=10.9cm, Shin=8.8cm
[L] Bone Length check rejected: Thigh=6.0cm, Shin=1.3cm
[R] Bone Length check rejected: Thigh=8.1cm, Shin=5.9cm
[L] Bone Length check rejected: Thigh=6.5cm, Shin=3.8cm
[R] Bone Length check rejected: Thigh=7.0cm, Shin=8.7cm
[L] Bone Length check rejected: Thigh=16.5cm, Shin=3.3cm
[R] Bone Length check rejected: Thigh=18.8cm, Shin=4.7cm
[L] Bone Length check rejected: Thigh=14.9cm, Shin=7.5cm
[R] Bone Length check rejected: Thigh=15.5cm, Shin=9.0cm
[L] Bone Length check rejected: Thigh=12.3cm, Shin=11.2cm

[L] Bone Length check rejected: Thigh=8.4cm, Shin=18.2cm
[R] Bone Length check rejected: Thigh=10.2cm, Shin=15.1cm
[L] Bone Length check rejected: Thigh=8.1cm, Shin=18.4cm
[R] Bone Length check rejected: Thigh=8.7cm, Shin=18.0cm
[L] Bone Length check rejected: Thigh=8.3cm, Shin=17.8cm
[R] Bone Length check rejected: Thigh=9.4cm, Shin=14.9cm
[L] Bone Length check rejected: Thigh=8.7cm, Shin=18.0cm
[R] Bone Length check rejected: Thigh=9.6cm, Shin=17.1cm
[L] Bone Length check rejected: Thigh=8.1cm, Shin=18.2cm
[R] Bone Length check rejected: Thigh=9.3cm, Shin=15.3cm
[L] Bone Length check rejected: Thigh=7.4cm, Shin=18.5cm
[R] Bone Length check rejected: Thigh=7.6cm, Shin=17.9cm
[L] Bone Length check rejected: Thigh=8.0cm, Shin=18.6cm
[R] Bone Length check rejected: Thigh=8.9cm, Shin=18.1cm
[L] Bone Length check rejected: Thigh=12.9cm, Shin=15.5cm
[R] Bone Length check rejected: Thigh=10.6cm, Shin=16.4cm
[L] Bone Length check rejected: Thigh=8.6cm, Shin=16.9cm
[R] Bone Length check reject

[L] Bone Length check rejected: Thigh=11.5cm, Shin=8.0cm
[R] Bone Length check rejected: Thigh=13.9cm, Shin=15.8cm
[L] Bone Length check rejected: Thigh=14.4cm, Shin=5.6cm
[R] Bone Length check rejected: Thigh=13.5cm, Shin=15.2cm
[L] Bone Length check rejected: Thigh=15.5cm, Shin=6.1cm
[R] Bone Length check rejected: Thigh=14.0cm, Shin=16.4cm
[L] Bone Length check rejected: Thigh=15.1cm, Shin=5.4cm
[R] Bone Length check rejected: Thigh=14.2cm, Shin=16.1cm
[L] Bone Length check rejected: Thigh=13.0cm, Shin=7.9cm
[R] Bone Length check rejected: Thigh=13.7cm, Shin=15.1cm
[L] Bone Length check rejected: Thigh=12.6cm, Shin=7.4cm
[R] Bone Length check rejected: Thigh=13.9cm, Shin=15.7cm
[L] Bone Length check rejected: Thigh=11.8cm, Shin=8.9cm
[R] Bone Length check rejected: Thigh=13.8cm, Shin=16.2cm
[L] Bone Length check rejected: Thigh=12.1cm, Shin=8.5cm
[R] Bone Length check rejected: Thigh=14.3cm, Shin=16.3cm
[L] Bone Length check rejected: Thigh=14.4cm, Shin=5.6cm
[R] Bone Length check r

[L] Bone Length check rejected: Thigh=20.2cm, Shin=9.4cm
[R] Bone Length check rejected: Thigh=15.7cm, Shin=11.4cm
[L] Bone Length check rejected: Thigh=20.7cm, Shin=7.5cm
[R] Bone Length check rejected: Thigh=18.8cm, Shin=10.6cm
[L] Bone Length check rejected: Thigh=13.3cm, Shin=9.8cm
[R] Bone Length check rejected: Thigh=17.0cm, Shin=8.1cm
[L] Bone Length check rejected: Thigh=18.8cm, Shin=8.7cm
[R] Bone Length check rejected: Thigh=18.1cm, Shin=9.0cm
[L] Bone Length check rejected: Thigh=19.2cm, Shin=9.3cm
[R] Bone Length check rejected: Thigh=18.0cm, Shin=7.2cm
[L] Bone Length check rejected: Thigh=11.9cm, Shin=14.7cm
[R] Bone Length check rejected: Thigh=11.0cm, Shin=13.7cm
[L] Bone Length check rejected: Thigh=10.0cm, Shin=16.5cm
[R] Bone Length check rejected: Thigh=8.1cm, Shin=11.6cm
[L] Bone Length check rejected: Thigh=10.0cm, Shin=15.7cm
[R] Bone Length check rejected: Thigh=7.8cm, Shin=12.8cm
[L] Bone Length check rejected: Thigh=9.6cm, Shin=17.2cm
[R] Bone Length check rej

[L] Bone Length check rejected: Thigh=9.0cm, Shin=17.6cm
[R] Bone Length check rejected: Thigh=7.7cm, Shin=18.1cm
[L] Bone Length check rejected: Thigh=10.4cm, Shin=17.1cm
[R] Bone Length check rejected: Thigh=9.8cm, Shin=17.5cm
[L] Bone Length check rejected: Thigh=9.7cm, Shin=12.0cm
[R] Bone Length check rejected: Thigh=9.6cm, Shin=10.1cm
[L] Bone Length check rejected: Thigh=6.4cm, Shin=14.8cm
[R] Bone Length check rejected: Thigh=9.3cm, Shin=13.6cm
[L] Bone Length check rejected: Thigh=12.2cm, Shin=17.3cm
[R] Bone Length check rejected: Thigh=14.2cm, Shin=11.3cm
[R] Bone Length check rejected: Thigh=17.8cm, Shin=12.0cm
[L] Bone Length check rejected: Thigh=13.1cm, Shin=14.8cm
[R] Bone Length check rejected: Thigh=12.6cm, Shin=14.9cm
[L] Bone Length check rejected: Thigh=10.4cm, Shin=15.3cm
[R] Bone Length check rejected: Thigh=8.7cm, Shin=17.2cm
[L] Bone Length check rejected: Thigh=6.5cm, Shin=20.1cm
[R] Bone Length check rejected: Thigh=7.1cm, Shin=18.6cm
[L] Bone Length check re

[L] Bone Length check rejected: Thigh=8.7cm, Shin=3.6cm
[R] Bone Length check rejected: Thigh=10.0cm, Shin=4.7cm
[R] Bone Length check rejected: Thigh=13.9cm, Shin=19.2cm
Safely releasing camera hardware...
Finished cleanup!
